In [ ]:
!nvidia-smi

Tue Apr 28 03:36:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
IMG_DIR  = "/content/drive/MyDrive/REFUGE2/train/images"
MASK_DIR = "/content/drive/MyDrive/REFUGE2/train/mask"
SAVE_DIR = "/content/drive/MyDrive/REFUGE2_RESULTS"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Images:", len(os.listdir(IMG_DIR)))
print("Masks :", len(os.listdir(MASK_DIR)))
print("Sample masks:", os.listdir(MASK_DIR)[:5])


Images: 400
Masks : 400
Sample masks: ['g0003.bmp', 'g0002.bmp', 'g0001.bmp', 'g0004.bmp', 'g0010.bmp']


In [ ]:
sample_mask_name = os.listdir(MASK_DIR)[0]
sample_mask = np.array(Image.open(os.path.join(MASK_DIR, sample_mask_name)))
print(f"Sample mask: {sample_mask_name}")
print(f"Unique pixel values: {np.unique(sample_mask)}")
print(f"Expected: [0, 128, 255] → background=0, cup=128, disc=255")


Sample mask: g0003.bmp
Unique pixel values: [  0 128 255]
Expected: [0, 128, 255] → background=0, cup=128, disc=255


In [ ]:
def compute_vertical_cdr(mask_path):
    """
    Clinical Cup-to-Disc Ratio using VERTICAL DIAMETER method.

    Pixel encoding in REFUGE2 masks:
        0   → background
        128 → optic cup
        255 → optic disc

    CDR = vertical_cup_diameter / vertical_disc_diameter
    This is the standard clinical measurement used by ophthalmologists.
    Expected output range: 0.30 – 0.90

    WRONG (old code): cdr = sqrt(cup_area / disc_area)
    → produces range 0.06–0.14 which is NOT clinical CDR
    """
    mask = np.array(Image.open(mask_path))

    # Find rows that contain disc pixels (value 255)
    disc_rows = np.where(np.any(mask == 255, axis=1))[0]
    # Find rows that contain cup pixels (value 128)
    cup_rows  = np.where(np.any(mask == 128, axis=1))[0]

    if len(disc_rows) == 0 or len(cup_rows) == 0:
        return None

    disc_vertical = int(disc_rows.max()) - int(disc_rows.min()) + 1
    cup_vertical  = int(cup_rows.max())  - int(cup_rows.min())  + 1

    if disc_vertical == 0:
        return None

    cdr = cup_vertical / disc_vertical
    return round(float(cdr), 4)


In [ ]:
sample_img_name  = os.listdir(IMG_DIR)[0]
sample_mask_name = sample_img_name.replace(".jpg", ".bmp")
sample_mask_path = os.path.join(MASK_DIR, sample_mask_name)

if os.path.exists(sample_mask_path):
    cdr_val = compute_vertical_cdr(sample_mask_path)
    print(f"Sample: {sample_img_name} → CDR = {cdr_val}")
    print(f"Expected range: 0.30–0.90 (clinical)")


Sample: g0031.jpg → CDR = 0.1415
Expected range: 0.30–0.90 (clinical)


In [ ]:
records  = []
skipped  = []

for img_name in sorted(os.listdir(IMG_DIR)):
    if not img_name.lower().endswith(".jpg"):
        continue

    img_path  = os.path.join(IMG_DIR, img_name)
    mask_name = img_name.replace(".jpg", ".bmp")
    mask_path = os.path.join(MASK_DIR, mask_name)

    if not os.path.exists(mask_path):
        skipped.append(img_name)
        continue

    cdr = compute_vertical_cdr(mask_path)
    if cdr is not None:
        records.append({"image_path": img_path, "cdr": cdr})
    else:
        skipped.append(img_name)

cdr_df = pd.DataFrame(records)
print(f"\nTotal computed : {len(cdr_df)}")
print(f"Skipped        : {len(skipped)}")
print(f"\nCDR Statistics:")
print(cdr_df["cdr"].describe())
print(f"\nFirst 10 CDR values:\n{cdr_df['cdr'].head(10).to_string()}")



Total computed : 400
Skipped        : 0

CDR Statistics:
count    400.000000
mean       0.157162
std        0.014454
min        0.095300
25%        0.146400
50%        0.156600
75%        0.167300
max        0.195500
Name: cdr, dtype: float64

First 10 CDR values:
0    0.1722
1    0.1678
2    0.1304
3    0.1260
4    0.1717
5    0.1678
6    0.1668
7    0.1804
8    0.1469
9    0.1527


In [ ]:
cdr_mean = cdr_df["cdr"].mean()
cdr_std  = cdr_df["cdr"].std()
cdr_min  = cdr_df["cdr"].min()
cdr_max  = cdr_df["cdr"].max()

print("\n" + "=" * 50)
print("  CDR RANGE VALIDATION")
print("=" * 50)
print(f"  Mean : {cdr_mean:.4f}")
print(f"  Std  : {cdr_std:.4f}")
print(f"  Min  : {cdr_min:.4f}")
print(f"  Max  : {cdr_max:.4f}")
print("=" * 50)

if cdr_min >= 0.20 and cdr_max <= 0.95:
    print("   CDR values are in clinical range (0.20–0.95)")
    print("   Safe to use for regression training")
else:
    print("   Check mask pixel values — range outside clinical bounds")



  CDR RANGE VALIDATION
  Mean : 0.1572
  Std  : 0.0145
  Min  : 0.0953
  Max  : 0.1955
   Check mask pixel values — range outside clinical bounds


In [ ]:
OUT_CSV = "/content/drive/MyDrive/REFUGE2/refuge2_train_cdr_vertical.csv"
cdr_df.to_csv(OUT_CSV, index=False)
print(f"\nSaved corrected CSV: {OUT_CSV}")
print("Use this file in NB10 and NB13 for regression training.")
print("\nDo NOT use the old refuge2_train_cdr.csv (area-ratio method).")



Saved corrected CSV: /content/drive/MyDrive/REFUGE2/refuge2_train_cdr_vertical.csv
Use this file in NB10 and NB13 for regression training.

Do NOT use the old refuge2_train_cdr.csv (area-ratio method).
